# Low-Bit Quantization

> A 14B model needs about 28 GB for BF16 weights and does not fit on a 16 GB GPU. At four bits per parameter, weights need about 7 GB. The arithmetic is simple; the real questions are why the model still works and where quality begins to fail.
>
> Decode is often limited by moving weights every step. Quantization reduces both stored memory and transferred bytes, improving capacity and TPOT.
>
> We cover INT4 scales and granularity; integer and floating formats; GPTQ, AWQ, and SmoothQuant; notation such as `W4A16`; deployment formats; and practical selection trade-offs.


## 1. Model Size and Precision

Inference memory is dominated by fixed model weights and a KV Cache that grows with context. Weight quantization targets the object that both occupies memory and moves every Decode step. The following chart compares weight footprints by precision.


In [ ]:
def weight_size_gb(params_billion, bits):
    """Estimate weight memory in GB from parameter count and bits per parameter."""
    return params_billion * 1e9 * bits / 8 / 1e9

for size in [7, 70]:
    for bits in [16, 8, 4]:
        print(f"{size:>2d}B @ {bits:2d}-bit -> {weight_size_gb(size, bits):6.1f} GB")

print()
print("Key observation: 70B in BF16 needs about 140 GB and does not fit on one GPU; at 4-bit it needs about 35 GB and does.")


In [ ]:
# Memory map: parameter count and precision determine how many weight bytes each Decode step must load
import matplotlib.pyplot as plt

bits_options = [16, 8, 4]
sizes = [7, 70]
xs = range(len(bits_options))
width = 0.35

plt.figure(figsize=(6.5, 3.5))
for j, size in enumerate(sizes):
    values = [weight_size_gb(size, b) for b in bits_options]
    plt.bar([x + (j - 0.5) * width for x in xs], values, width=width,
            label=f"{size}B params")
plt.xticks(list(xs), [f"{b}-bit" for b in bits_options])
plt.ylabel("weight size (GB)")
plt.title("Smaller dtype = less memory to load every decode step")
plt.legend()
plt.show()


The chart explains why quantization is attractive, but creates a deeper question: how can 16 INT4 levels replace the many values available in 16-bit floating point without destroying the model? We first define the actual mapping operation.


## 2. From Floating Point to INT4

INT4 provides 16 codes; symmetric quantization commonly uses -7 through 7. Quantization maps a continuous value range to fixed levels rather than merely rounding decimals to whole numbers. The interval is the **scale**.

For scale $1/7\approx0.143$, `-0.72` maps to -5 and reconstructs near -0.714, while `0.63` maps to 4 and reconstructs near 0.571. Each value incurs at most about half a scale interval of error. Millions of small errors can accumulate, motivating methods that reduce output damage.


In [ ]:
import numpy as np

x = np.array([-1.0, -0.72, -0.31, 0.0, 0.18, 0.63, 1.0], dtype=np.float32)
qmax = 7
scale = np.max(np.abs(x)) / qmax
q = np.round(x / scale).clip(-qmax, qmax).astype(np.int32)
x_hat = q * scale

print("scale:", round(float(scale), 4))
print("float:", x)
print("INT4 :", q)
print("dequantized:", np.round(x_hat, 3))
print()
print(f"Key observation: maximum error {np.max(np.abs(x - x_hat)):.4f} < scale/2 = {scale/2:.4f}")
print("Every value moves to its nearest grid point; quantization error is the accumulated effect of millions of these moves.")


## 3. Quantization Granularity

One scale for an entire layer assumes similar ranges everywhere. Real tensors contain channels with very different ranges and occasional outliers. Finer granularity confines an outlier's influence:

```text
per-tensor  : one scale for the tensor
per-channel : one scale per channel
per-group   : one scale per group, often 128 values in modern 4-bit formats
```

Finer scales reduce error but add metadata and kernel complexity. We compare them on synthetic weights containing an outlier.


In [ ]:
np.random.seed(7)
W = np.random.randn(4, 16).astype(np.float32) * 0.25
W[1] *= 8  # channel 1 is an outlier, with a range enlarged by 8x

def qdq_tensor(a):
    """Use one scale for the entire tensor."""
    s = max(np.max(np.abs(a)) / 7, 1e-12)
    q = np.round(a / s).clip(-7, 7)
    return q * s

def qdq_channel(a):
    """Use one scale per channel (per row)."""
    s = np.maximum(np.max(np.abs(a), axis=1, keepdims=True) / 7, 1e-12)
    q = np.round(a / s).clip(-7, 7)
    return q * s

def qdq_group(a, group=4):
    """Give each group of four adjacent elements its own scale."""
    out = np.zeros_like(a)
    for r in range(a.shape[0]):
        for c0 in range(0, a.shape[1], group):
            block = a[r, c0:c0 + group]
            s = max(np.max(np.abs(block)) / 7, 1e-12)
            out[r, c0:c0 + group] = np.round(block / s).clip(-7, 7) * s
    return out

errs = {}
for name, fn in [("per-tensor", qdq_tensor), ("per-channel", qdq_channel),
                 ("per-group(4)", lambda a: qdq_group(a, 4))]:
    errs[name] = float(np.mean(np.abs(W - fn(W))))
    print(f"{name:<14} MAE = {errs[name]:.4f}")

print()
print("Key observation: with an outlier channel, finer granularity prevents that outlier from degrading the other channels as much.")


In [ ]:
# Compare errors for the same weights at three granularities: greener means lower error
import matplotlib.pyplot as plt

plt.figure(figsize=(5.5, 3.2))
plt.bar(errs.keys(), errs.values(), color=["tab:red", "tab:orange", "tab:green"])
plt.ylabel("MAE (lower is better)")
plt.title("Finer granularity -> smaller quantization error")
plt.show()


## 4. Weights and Activations

**Activations** are layer inputs that change with each prompt. Quantization notation names both sides:

```text
W4A16 = 4-bit weights, 16-bit activations
W8A8  = 8-bit weights, 8-bit activations
```

Weights are fixed and can be calibrated offline. Activations change online and often contain large outliers in a few channels. This makes weight-only quantization easier and motivates algorithms that address different sources of error.


## 5. GPTQ, AWQ, and SmoothQuant

Round-To-Nearest (RTN) is the baseline. At 8 bits it often works well; at 4 bits accumulated error becomes more important.

**GPTQ** quantizes weights column by column and uses approximate second-order information to compensate errors in unquantized columns, keeping outputs near the original model.

**AWQ** observes that channels with large activations amplify weight error. It protects these salient channels through activation-aware scaling without full backpropagation.

**SmoothQuant** moves difficulty from activations to weights through an equivalent channel scaling: divide activation channels and multiply corresponding weight channels. The mathematical output is unchanged, but both tensors become easier to quantize.

| Algorithm | Problem addressed | Route |
|:---|:---|:---|
| GPTQ | Accumulated 4-bit weight error | Weight-only, second-order compensation |
| AWQ | Damage to salient channels | Weight-only, activation-aware protection |
| SmoothQuant | Activation outliers | W8A8, migrate numerical difficulty |

These are algorithms, not file formats; the same nominal 4-bit width can have different quality.


## 6. PTQ, QAT, and KV Cache Quantization

**PTQ** (Post-Training Quantization) transforms an existing checkpoint, often with a calibration set and no full training. **QAT** (Quantization-Aware Training) inserts fake quantize/dequantize operations during training so the model adapts to low-bit error. PTQ is usually tried first; QAT costs more but may recover quality.

KV Cache quantization targets a different object from `W4A16`. At long contexts and high concurrency, storing KV as FP8 or INT8 can halve its memory. Engine settings often expose this as `kv_cache_dtype=fp8`.


## 7. Floating Formats: FP8 and FP4

INT levels are uniformly spaced, while weights often cluster near zero with a few distant outliers. Floating formats use exponent and mantissa to create dense levels near zero and coarser levels farther away:

$$\underbrace{\pm}_{sign}\;\underbrace{2^e}_{exponent}\;\underbrace{\times1.m}_{mantissa}$$

| Format | Bits | Exponent / mantissa | Typical setting |
|:---|---:|:---|:---|
| FP16 | 16 | 5 / 10 | Traditional mixed precision |
| BF16 | 16 | 8 / 7 | Modern training default |
| FP8 E4M3 | 8 | 4 / 3 | Precision-oriented training/inference |
| FP8 E5M2 | 8 | 5 / 2 | Range-oriented values |
| FP4 E2M1 | 4 | 2 / 1 | New hardware paths |

Hopper GPUs accelerated FP8, while Blackwell adds FP4-oriented paths such as NVFP4 and MXFP4 with block scales. The experiment compares uniform INT4 and nonuniform FP4 reconstruction on normally distributed weights.


In [ ]:
# The 16 FP4 E2M1 levels: the exponent creates denser levels near zero
# Positive levels are {0, 0.5, 1, 1.5, 2, 3, 4, 6}, mirrored on the negative side
fp4_grid = np.array([-6, -4, -3, -2, -1.5, -1, -0.5,
                     0, 0.5, 1, 1.5, 2, 3, 4, 6])

def qdq_fp4(a):
    """Quantize to E2M1 floating-point levels, then reconstruct."""
    s = np.max(np.abs(a)) / 6.0
    z = a / s
    idx = np.abs(z[..., None] - fp4_grid).argmin(axis=-1)
    return fp4_grid[idx] * s

def qdq_int4(a):
    """Baseline: symmetric INT4 with 15 uniform levels (-7 through +7, conventionally omitting -8)."""
    s = np.max(np.abs(a)) / 7.0
    return np.round(a / s).clip(-7, 7) * s

np.random.seed(0)
W = np.random.randn(1000, 100).astype(np.float32) * 0.5

err_int4 = float(np.mean(np.abs(W - qdq_int4(W))))
err_fp4 = float(np.mean(np.abs(W - qdq_fp4(W))))
print(f"INT4 (uniform 15 levels)   MAE = {err_int4:.4f}")
print(f"FP4 E2M1 (float 16 levels) MAE = {err_fp4:.4f}")
print()
print("Key observation: at the same 4 bits, floating-point levels better fit a distribution concentrated near zero.")
print("This is the underlying reason FP8 and FP4 are useful on newer hardware.")


In [ ]:
# Left: weight histogram, dense in the middle and sparse at the tails; right: errors of the two formats
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].hist(W.flatten(), bins=80, color="tab:gray", alpha=0.7, density=True)
axes[0].set_title("Weight distribution: dense near zero")
axes[0].set_xlabel("value")
axes[0].set_ylabel("density")

axes[1].bar(["INT4\n(uniform)", "FP4 E2M1\n(float)"], [err_int4, err_fp4],
            color=["tab:red", "tab:green"])
axes[1].set_ylabel("MAE (lower is better)")
axes[1].set_title("Same 4 bits, different grid shape")
plt.tight_layout()
plt.show()


## 8. GGUF and the llama.cpp Ecosystem

**GGUF** packages weights, vocabulary, and configuration in one file for llama.cpp and tools such as Ollama and LM Studio. A name such as `Q4_K_M` reads as four-bit main weights, K-quant block structure, and a Medium mix that keeps selected tensors at higher precision.

K-quant uses block scales, the same principle as per-group quantization. Variants range from compact `Q2_K` through common `Q4_K_M` to near-lossless `Q8_0`. An importance matrix (`imatrix`) uses calibration text to concentrate error in less important positions, resembling AWQ's protection principle.

GGUF is especially useful for CPU, Apple Silicon, edge devices, or quick local runs. High-concurrency GPU serving more commonly uses GPTQ, AWQ, or FP8 with vLLM/SGLang. Formats are tied to runtimes rather than universally interchangeable.


## 9. Choosing a Quantization Scheme

| Deployment target | Common format | Runtime |
|:---|:---|:---|
| NVIDIA GPU high-concurrency serving | GPTQ/AWQ INT4, FP8 checkpoint | vLLM / SGLang / TensorRT-LLM |
| Blackwell hardware | NVFP4 / MXFP4 | TensorRT-LLM / vLLM |
| Quick low-memory experiments | bitsandbytes NF4 | Transformers |
| CPU / Mac / edge | GGUF such as Q4_K_M | llama.cpp / Ollama / LM Studio |
| Long-context concurrency | W8A8 or FP8 weights plus FP8 KV | vLLM / SGLang |

First choose the runtime and check its support matrix, then select format and bit width. A GPTQ checkpoint and a GGUF file are both “4-bit” but belong to different execution paths.


## 10. Quantization in Practice

Three common paths produce different deployment artifacts:

| Path | Tool | Output | Runtime |
|:---|:---|:---|:---|
| GPTQ / FP8 | llm-compressor | Hugging Face checkpoint directory | vLLM / SGLang |
| AWQ | AutoAWQ | Hugging Face checkpoint directory | vLLM / SGLang |
| GGUF | llama.cpp conversion and quantization tools | One `.gguf` file | llama.cpp / Ollama |

### Path 1: GPTQ and FP8 with llm-compressor

`llm-compressor` uses a shared `oneshot` interface with different recipes. `W4A16` means four-bit weights with 16-bit activations, and sensitive tensors such as `lm_head` are often skipped. `FP8_DYNAMIC` quantizes weights offline while choosing activation scales dynamically at runtime, addressing input-dependent activation ranges.


### Path 2: AWQ with AutoAWQ

`w_bit=4` sets bit width, `q_group_size=128` sets group granularity, and `zero_point=True` selects asymmetric quantization. Calibration estimates channel importance; domain-representative `calib_data` improves protection of channels important to real traffic.


### Path 3: GGUF with llama.cpp

Convert Hugging Face weights to an unquantized F16 GGUF, then quantize that file to `Q4_K_M` or another level. Retaining the F16 intermediate permits several comparisons. An optional `llama-imatrix` pass calculates importance from calibration text before final quantization.


### Running the Result

GPTQ/AWQ checkpoints are generally detected by compatible GPU engines; a BF16 checkpoint may also be quantized to FP8 while loading for a quick experiment. llama.cpp serves GGUF through an OpenAI-compatible API.

Online FP8 is convenient for exploration, while production should prefer calibrated offline checkpoints when quality matters.


## 11. Benefits and Costs of Quantization

**Benefits:** smaller memory, less Decode bandwidth, potentially higher throughput, and capacity for larger models, longer contexts, or more concurrency. New hardware accelerates FP8/FP4 directly.

**Costs:** some quality loss, often more visible in math and code. Smaller models are generally more sensitive. Group scales add metadata, and unsupported kernels may dequantize before computation.

Deployment therefore requires a quality benchmark after quantization. Only workload-specific evaluation can decide whether the saved memory is worth the measured degradation.


## Summary

- [ ] Be able to write the formulas for symmetric and asymmetric quantization, and explain the role of the zero point
- [ ] Understand the tradeoff between precision and parameter overhead across per-tensor, per-channel, and per-group granularities
- [ ] Be able to explain why activation quantization is harder than weight quantization, and how outlier channels break the scale
- [ ] Be able to state in one sentence the core idea of GPTQ (compensate error with the Hessian) and AWQ (protect important channels by scaling)
- [ ] Know that the current mainstream 4-bit LLM inference configuration is weight-only quantization (such as Q4_K_M, AWQ)


## Exercises

The three exercises below help you put the core algorithms of this section into practice. You can ask an AI to help break down steps or check ideas, but it is not recommended to let an AI write the complete answer directly — the details of quantization only stick after you have written them once yourself.

**Exercise 1: Hand-calculate asymmetric quantization**

Given the vector $x = [0.5, 1.0, 1.5, 2.0, 2.5]$, perform INT4 asymmetric quantization (uint4, range $[0, 15]$). By hand, compute $x_{\min}$, $x_{\max}$, scale, and zero_point, write out the quantized integer and the dequantized result for each element, and compute the MAE.

Hint: first compute $s = (x_{\max} - x_{\min}) / 15$, then use $q = \text{round}((x - x_{\min}) / s)$.

**Exercise 2: Implement per-channel symmetric quantization**

Implement a function `quantize_per_channel(W, n_bits=4)` whose input is a weight matrix of shape `(out_features, in_features)`, and which independently performs symmetric quantization on each row. Return the quantized-then-dequantized floating-point matrix.


### Exercise 1: Implement Per-Group Quantization

Quantize each adjacent group with its own scale, reconstruct it, and compare error with per-tensor quantization.

Hint: `scale=np.max(np.abs(block))/7`, then round, clip to `[-7,7]`, and multiply back.


In [ ]:
# Exercise 1: fill in per-group quantization

np.random.seed(0)
W_test = np.random.randn(8, 32).astype(np.float32)
W_test[0] *= 10  # create an outlier row

def qdq_group_cols(a, group=8):
    """Quantize and reconstruct each row in adjacent groups, using one scale per group."""
    out = np.zeros_like(a)
    for r in range(a.shape[0]):
        for c0 in range(0, a.shape[1], group):
            block = a[r, c0:c0 + group]
            # TODO: replace the triple-quoted text with your code
            """Compute the group scale, round and clip, rescale, and write into out[r, c0:c0+group]."""
    return out

err_tensor = np.mean(np.abs(W_test - qdq_tensor(W_test)))
err_group = np.mean(np.abs(W_test - qdq_group_cols(W_test)))
assert err_group < err_tensor, (err_group, err_tensor)
print("✅ Exercise 1 passed: an outlier affects only its own group, reducing overall error.")


### Exercise 2: Asymmetric Quantization with a Zero Point

For mostly positive activations, map `[min,max]` to `[0,14]` and record where zero lies.

Hint: `scale=(max-min)/14`, `zero_point=round(-min/scale)`, quantize with `round(x/scale)+zero_point`, then reconstruct `(q-zero_point)*scale`.


In [ ]:
# Exercise 2: fill in asymmetric quantization

x = np.array([2.0, 2.5, 3.0, 3.5, 4.0], dtype=np.float32)  # entirely on the positive axis
scale = (x.max() - x.min()) / 14
zero_point = int(np.round(-x.min() / scale))

def qdq_asym(a):
    """Map [min, max] to [0, 14], then dequantize."""
    # TODO: replace the triple-quoted text with your code
    """Set q = round(a/scale) + zero_point, clip to [0,14], then dequantize."""

x_hat = qdq_asym(x)
assert np.max(np.abs(x - x_hat)) <= scale / 2 + 1e-6, (x, x_hat)
print("✅ Exercise 2 passed: asymmetric quantization uses the full range and keeps error within half a grid interval.")


### Exercise 3: Read a Quantized Model Name

Parse names such as `Qwen2.5-7B-Instruct-GPTQ-Int4` into method and bit width.

Hint: detect `GPTQ`, `AWQ`, or `GGUF`; explicit `Int8` means eight bits, otherwise many named variants default to four.


In [ ]:
# Exercise 3: parse a model name

def parse_quant_name(name):
    """Extract {'bits': int, 'method': str} from a quantized model name."""
    # TODO: replace the triple-quoted text with your code
    """Read the digit from Int8 or Int4 (default 4); choose GPTQ, AWQ, or GGUF as the method."""

info = parse_quant_name("Qwen2.5-7B-Instruct-GPTQ-Int4")
assert info == {"bits": 4, "method": "GPTQ"}, info
info2 = parse_quant_name("Llama-3-8B-AWQ")
assert info2 == {"bits": 4, "method": "AWQ"}, info2
print("✅ Exercise 3 passed: you can identify the method and precision from a quantized model name.")
